In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import os

### Downloading dataset from kaggle

In [2]:
!kaggle datasets download -d tanishgupta26/famous-personalities-image-dataset
!unzip famous-personalities-image-dataset.zip

'kaggle' is not recognized as an internal or external command,
operable program or batch file.
'unzip' is not recognized as an internal or external command,
operable program or batch file.


In [3]:
dataset_path = "Dataset/Dataset"
print(os.listdir(dataset_path))

['Dalai_Lama', 'Melinda_Gates', 'Virat_Kohli', 'Bill_Gates', 'Vikas_Khanna', 'Indira_Nooyi', 'Narendra_Modi', 'Sundar_Pichai', 'Barack_Obama', 'Anushka_Sharma']


### Data Preprocessing

About `ImageDataGenerator` object, which is commonly used in deep learning for image augmentation and preprocessing. Here's a breakdown of each part:

*   **`datagen = ImageDataGenerator(...)`**: This creates an instance of the `ImageDataGenerator` class, likely from a library like Keras or TensorFlow. This object is designed to generate batches of augmented image data (and their corresponding labels) on the fly, which is very useful for training deep learning models.

*   **`rescale=1./255`**: This is a common preprocessing step. Image pixel values typically range from 0 to 255. By rescaling them to `1./255`, the pixel values will be normalized to a range between 0 and 1. This helps in faster convergence during model training.

*   **`validation_split=0.2`**: This parameter indicates that 20% of the data will be reserved for validation. When you later use methods like `flow_from_directory` with this generator, you can specify `subset='training'` or `subset='validation'` to get respective data subsets.

*   **`rotation_range=20`**: This applies random rotations to images. Images will be randomly rotated by an angle within the range of -20 to +20 degrees. This helps the model generalize better to images that might be slightly rotated.

*   **`zoom_range=0.2`**: This applies random zooming to images. Images will be randomly zoomed in or out by up to 20%. For example, a zoom factor of 0.8 to 1.2 is applied (1-0.2 to 1+0.2). This makes the model robust to variations in object size.

*   **`horizontal_flip=True`**: This randomly flips images horizontally. This is a common augmentation technique, especially for objects that are symmetrical or whose left-right orientation doesn't change their identity (e.g., faces, animals).

In [4]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2,
    rotation_range=25,
    zoom_range=0.2,
    horizontal_flip=True
)

These two code blocks use the `flow_from_directory` method of the `ImageDataGenerator` object to create data generators for your training and validation sets. Let's break down each part:

*   **`datagen.flow_from_directory(...)`**: This method is designed to read images from subdirectories, where each subdirectory represents a different class. It efficiently loads and augments images in batches.

*   **`dataset_path`**: This is the path to your main dataset directory, which you previously defined. The `ImageDataGenerator` expects to find subdirectories within this path, with each subdirectory name being a class label (e.g., `dataset/Alejandro Toledo`, `dataset/Alvaro Uribe`).

*   **`target_size=(224,224)`**: This resizes all images to 224x224 pixels. This is a common practice to ensure all input images have a consistent size for deep learning models.

*   **`batch_size=32`**: This specifies the number of images that will be yielded in each batch by the generator. Training deep learning models often involves processing data in small batches rather than all at once.

*   **`subset="training"`**: For the `train` generator, this tells the `flow_from_directory` method to use the portion of the data designated for training, based on the `validation_split=0.2` you set earlier in the `ImageDataGenerator`.

*   **`subset="validation"`**: Similarly, for the `val` generator, this tells the method to use the 20% of the data reserved for validation.

In essence, these lines create two efficient pipelines: one for feeding augmented training images to your model and another for feeding augmented validation images.

In [5]:
train = datagen.flow_from_directory(
    dataset_path,
    target_size=(224,224),
    batch_size=32,
    subset="training"
)

val = datagen.flow_from_directory(
    dataset_path,
    target_size=(224,224),
    batch_size=32,
    subset="validation"
)

Found 4448 images belonging to 10 classes.
Found 1109 images belonging to 10 classes.


### MobileNet

**MobileNet** is a family of lightweight, efficient, and open-source Convolutional Neural Networks (CNNs) developed by Google, specifically designed for computer vision tasks on mobile and edge devices. It utilizes depthwise separable convolutions to minimize computational power and model size, making it ideal for real-time applications like classification, object detection, and segmentation on devices with limited resources

USING THIS MODEL FOR OUR PROJECT

In [6]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers, models

base_model = MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_shape=(224,224,3)
)

# fine-tune
base_model.trainable = True

for layer in base_model.layers[:-25]:
    layer.trainable = False

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [7]:
x = base_model.output
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(128, activation='relu')(x)
x = layers.Dropout(0.5)(x)

output = layers.Dense(train.num_classes, activation='softmax')(x)

model = models.Model(inputs=base_model.input, outputs=output)

### Model Training

In [8]:
from tensorflow.keras.optimizers import Adam

model.compile(
    optimizer=Adam(learning_rate=0.0001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

history = model.fit(
    train,
    validation_data=val,
    epochs=10
)

Epoch 1/10
139/139 ━━━━━━━━━━━━━━━━━━━━ 386s 3s/step - accuracy: 0.3919 - loss: 1.8192 - val_accuracy: 0.5780 - val_loss: 1.2620
Epoch 2/10
139/139 ━━━━━━━━━━━━━━━━━━━━ 365s 3s/step - accuracy: 0.6518 - loss: 1.0802 - val_accuracy: 0.7232 - val_loss: 0.8770
Epoch 3/10
139/139 ━━━━━━━━━━━━━━━━━━━━ 369s 3s/step - accuracy: 0.7304 - loss: 0.8089 - val_accuracy: 0.7115 - val_loss: 0.8513
Epoch 4/10
139/139 ━━━━━━━━━━━━━━━━━━━━ 364s 3s/step - accuracy: 0.7880 - loss: 0.6488 - val_accuracy: 0.7385 - val_loss: 0.8837
Epoch 5/10
139/139 ━━━━━━━━━━━━━━━━━━━━ 350s 3s/step - accuracy: 0.8159 - loss: 0.5465 - val_accuracy: 0.7385 - val_loss: 0.8668
Epoch 6/10
139/139 ━━━━━━━━━━━━━━━━━━━━ 365s 3s/step - accuracy: 0.8460 - loss: 0.4753 - val_accuracy: 0.7583 - val_loss: 0.7793
Epoch 7/10
139/139 ━━━━━━━━━━━━━━━━━━━━ 365s 3s/step - accuracy: 0.8606 - loss: 0.4141 - val_accuracy: 0.6943 - val_loss: 1.1654
Epoch 8/10
139/139 ━━━━━━━━━━━━━━━━━━━━ 352s 3s/step - accuracy: 0.8847 - loss: 0.3500 - val_accu

In [9]:
model.evaluate(val)

35/35 ━━━━━━━━━━━━━━━━━━━━ 58s 2s/step - accuracy: 0.7394 - loss: 0.9419


[0.9419435262680054, 0.7394048571586609]

### testing

In [19]:
from tensorflow.keras.preprocessing import image

img = image.load_img("/content/images (2).jpg", target_size=(224,224))
img = image.img_to_array(img)/255.0
img = np.expand_dims(img, axis=0)

pred = model.predict(img)
class_names = list(train.class_indices.keys())

print("Prediction:", class_names[np.argmax(pred)])

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 149ms/step
Prediction: Dalai_Lama


### saving the model

In [21]:
model.save("model_1.h5")